# Graph-Routed Atlas-DeePC

This notebook records the baseline fair-IID Reacher method. The controller is **DeePC, not MPC**: every optimized past/future trajectory is represented by a linear combination of offline behavioral data. A chart-transition graph chooses waypoint references, but it never generates control actions.

The controller always uses **48 physical-configuration charts** and cached rank-60 DeePC matrices. At every control step, the current chart selects the behavioral matrix and the transition graph selects the next chart-medoid waypoint. There is no contextual rescue, online nearest-trajectory retrieval, fitted dynamics model, inverse kinematics, or feedback fallback.


In [ ]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/atlas_deepc_mpl")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

import routed_deepc_experiment as experiment

DATA_PATH = ROOT / "data" / "select_iid_offline_hankel.npy"
RESULT_ROOT = ROOT / "experiments/routed_deepc_iid"
ATLAS_DIR = RESULT_ROOT / "clustering_jointconfig48_fresh211_310"
DEV_BASELINE_DIR = RESULT_ROOT / "clustering_jointconfig48_benchmark111_210"
EVAL_BASELINE_DIR = RESULT_ROOT / "clustering_jointconfig48_baseline_cache"
VIDEO_DIR = ATLAS_DIR

required_paths = [DATA_PATH, ATLAS_DIR / "behavioral_atlas.npz",
                  DEV_BASELINE_DIR / "summary.json", EVAL_BASELINE_DIR / "summary.json"]
assert all(path.exists() for path in required_paths)
print("Project root:", ROOT)


## 1. Local behavioral DeePC formulation

Let $X_k$ contain the complete input-output Hankel columns assigned to the current chart $k$. We compute its center $\bar h_k$ and cached rank-60 left-SVD basis $V_{k,60}$. Every prediction satisfies

$$
\begin{bmatrix}u_{\mathrm{ini}} \\ u \\ y_{\mathrm{ini}} + \sigma \\ y\end{bmatrix}
= \bar h_k + V_{k,60}g.
$$

The matrix always contains the complete $U_p,U_f,Y_p,Y_f$ blocks from one offline chart. The graph changes the waypoint reference and the measured configuration changes the active chart, but neither operation supplies actions. The same fixed DeePC QP computes every control input, and no state-space matrices are fitted.


In [ ]:
dev_summary = json.loads((DEV_BASELINE_DIR / "summary.json").read_text())
evaluation_summary = json.loads((EVAL_BASELINE_DIR / "summary.json").read_text())
video_summary = json.loads((ATLAS_DIR / "summary.json").read_text())
evaluation_runs = evaluation_summary["results"]
development_seeds = {int(row["seed"]) for row in dev_summary["results"]}
evaluation_by_seed = {int(row["seed"]): row for row in evaluation_runs}
assert len(dev_summary["results"]) == len(development_seeds) == 100
assert len(evaluation_runs) == len(evaluation_by_seed), "duplicate evaluation seeds"
evaluation_seeds = sorted(evaluation_by_seed)
assert evaluation_seeds, "evaluation panel is empty"
evaluation_runs = [evaluation_by_seed[seed] for seed in evaluation_seeds]
development_overlap = sorted(development_seeds.intersection(evaluation_seeds))
if development_overlap:
    print(f"WARNING: {len(development_overlap)} evaluation seeds overlap the development panel; do not call them held out.")
config = experiment.RoutedDeePCConfig(**evaluation_summary["config"])
H = np.load(DATA_PATH, mmap_mode="r")
atlas = experiment.load_atlas(ATLAS_DIR / "behavioral_atlas.npz")
selection_atlas = experiment.load_atlas(DEV_BASELINE_DIR / "behavioral_atlas.npz")
for key in ["labels", "centers", "transition_counts", "medoid_states"]:
    assert np.array_equal(atlas[key], selection_atlas[key])
chart0_matrix, chart0_columns = experiment.chart_matrix(H, atlas, [0], config)

assert config.clusters == 48
assert config.clustering_method == "minibatch_kmeans"
assert config.clustering_feature_mode == "plant_joint_configuration"
assert config.clustering_input_weight == 0.0
assert config.formulation == "svd_current"
assert config.svd_rank == 60
assert chart0_matrix["basis"].shape[1] == 60
print("Offline Hankel shape:", H.shape)
print("Fixed physical-configuration charts:", config.clusters)
print("Chart-assignment feature: latest periodic joint configuration (6 active coordinates)")
print("Input weight in chart assignment:", config.clustering_input_weight)
print("Chart 0 offline columns:", chart0_columns.size)
print("Chart 0 affine basis shape:", chart0_matrix["basis"].shape)
print("Controller: cached current-chart Atlas-DeePC only")
print(f"Evaluation panel: {len(evaluation_seeds)} seeds ({evaluation_seeds[0]}-{evaluation_seeds[-1]})")
print("Fitted dynamics model: False")


## 2. Inspecting the constructed physical-configuration atlas

A chart is defined by K-means on the latest periodic joint configuration

$$
\phi(y_t)=[\cos q_1,\cos q_2,\sin q_1,\sin q_2,\cos(q_1+q_2),\sin(q_1+q_2)].
$$

Inputs, velocities, fingertip coordinates, and the preceding output are not used to assign charts. Each panel below overlays 32 randomly sampled chart members in a shallow color and draws the chart medoid boldly. This is the updated 48-chart version of the configuration-cloud plot.


In [ ]:
from matplotlib.collections import LineCollection

chart_sizes = np.bincount(atlas["labels"].astype(int), minlength=config.clusters)
medoid_states = np.asarray(atlas["medoid_states"], dtype=float)
chart_colors = plt.cm.turbo(np.linspace(0.0, 1.0, config.clusters))
chart_output_states = H[experiment.pipe.M * experiment.pipe.L:].reshape(
    experiment.pipe.L, experiment.pipe.P, H.shape[1]
)[experiment.pipe.T_INI - 1].T
cloud_members_per_chart = 32
cloud_rng = np.random.default_rng(17)

# Reacher-v5's output geometry is approximately 0.10 m to the elbow and
# 0.11 m from the elbow to the reported fingertip site.
link_1, link_2 = 0.10, 0.11
elbows = link_1 * np.column_stack([medoid_states[:, 0], medoid_states[:, 2]])
fk_fingertips = elbows + link_2 * np.column_stack([medoid_states[:, 8], medoid_states[:, 9]])
stored_fingertips = medoid_states[:, 6:8]
fk_error = np.linalg.norm(fk_fingertips - stored_fingertips, axis=1)

fig, axes = plt.subplots(6, 8, figsize=(16, 12), constrained_layout=True)
for chart_id, ax in enumerate(axes.flat):
    elbow = elbows[chart_id]
    fingertip = stored_fingertips[chart_id]
    color = chart_colors[chart_id]
    member_indices = np.flatnonzero(atlas["labels"] == chart_id)
    sampled_indices = cloud_rng.choice(
        member_indices, size=min(cloud_members_per_chart, member_indices.size), replace=False
    )
    cloud_states = chart_output_states[sampled_indices]
    cloud_elbows = link_1 * np.column_stack([cloud_states[:, 0], cloud_states[:, 2]])
    cloud_fingertips = cloud_states[:, 6:8]
    cloud_segments = np.stack([
        np.zeros_like(cloud_elbows), cloud_elbows, cloud_fingertips
    ], axis=1)
    ax.add_collection(LineCollection(
        cloud_segments, colors=[color], linewidths=1.05, alpha=0.24, zorder=1
    ))
    ax.scatter(cloud_fingertips[:, 0], cloud_fingertips[:, 1], s=7,
               color=color, alpha=0.32, linewidth=0, zorder=1)
    ax.plot([0, elbow[0], fingertip[0]], [0, elbow[1], fingertip[1]],
            color=color, linewidth=3.0, solid_capstyle="round", zorder=3)
    ax.scatter([0], [0], s=18, color="#111827", zorder=3)
    ax.scatter([elbow[0]], [elbow[1]], s=20, color="#374151", zorder=3)
    ax.scatter([fingertip[0]], [fingertip[1]], s=28, color=color,
               edgecolor="#111827", linewidth=0.5, zorder=4)
    ax.axhline(0, color="#d1d5db", linewidth=0.5, zorder=0)
    ax.axvline(0, color="#d1d5db", linewidth=0.5, zorder=0)
    ax.set_xlim(-0.225, 0.225)
    ax.set_ylim(-0.225, 0.225)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Chart {chart_id} | n={chart_sizes[chart_id]}", fontsize=7, color="#111827")
    for spine in ax.spines.values():
        spine.set_color(color)
        spine.set_linewidth(1.5)
fig.suptitle("Physical-configuration chart clouds with bold medoids", fontsize=16)
medoid_plot_path = RESULT_ROOT / "physical48_chart_configuration_clouds.png"
fig.savefig(medoid_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Medoid forward-kinematics consistency: max {1000 * fk_error.max():.3f} mm")
print(f"Chart occupancy: min {chart_sizes.min()}, median {np.median(chart_sizes):.0f}, max {chart_sizes.max()} windows")


In [ ]:
Y_windows = H[experiment.pipe.M * experiment.pipe.L:].reshape(
    experiment.pipe.L, experiment.pipe.P, H.shape[1]
)
offline_fingertips = Y_windows[experiment.pipe.T_INI - 1, 6:8].T
workspace_sample = np.linspace(0, offline_fingertips.shape[0] - 1, 8000, dtype=int)
elbow_up = medoid_states[:, 3] >= 0.0  # sign of sin(q2)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(offline_fingertips[workspace_sample, 0], offline_fingertips[workspace_sample, 1],
           s=3, color="#9ca3af", alpha=0.12, label="sampled offline states")
for mask, marker, label in [(elbow_up, "o", "medoid: sin(q2) >= 0"),
                            (~elbow_up, "s", "medoid: sin(q2) < 0")]:
    ax.scatter(stored_fingertips[mask, 0], stored_fingertips[mask, 1],
               c=chart_colors[mask], marker=marker, s=80, edgecolor="#111827",
               linewidth=0.7, label=label, zorder=3)
for chart_id, (x, y) in enumerate(stored_fingertips):
    ax.annotate(str(chart_id), (x, y), xytext=(3, 3), textcoords="offset points", fontsize=7)
ax.add_patch(plt.Circle((0, 0), link_1 + link_2, fill=False, linestyle="--",
                        color="#6b7280", linewidth=1, label="kinematic reach"))
ax.scatter([0], [0], color="#111827", s=35, label="arm base", zorder=4)
ax.set_aspect("equal")
ax.set_xlim(-0.225, 0.225)
ax.set_ylim(-0.225, 0.225)
ax.set_xlabel("fingertip x (m)")
ax.set_ylabel("fingertip y (m)")
ax.set_title("Physical workspace coverage of chart medoids")
ax.grid(alpha=0.2)
ax.legend(loc="upper right", fontsize=8)
workspace_plot_path = RESULT_ROOT / "atlas_workspace_coverage.png"
fig.savefig(workspace_plot_path, dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.decomposition import PCA

clustering_features, _, _ = experiment.behavioral_features(
    H,
    config.clustering_input_weight,
    config.clustering_feature_mode,
    config.clustering_configuration_weight,
    config.clustering_velocity_weight,
    config.clustering_fingertip_weight,
)
active_feature_mask = clustering_features.std(axis=0) > 1e-10
gate_features = clustering_features[:, active_feature_mask]
gate_centers = atlas["centers"][:, active_feature_mask]
pca = PCA(n_components=2)
gate_pca = pca.fit_transform(gate_features)
center_pca = pca.transform(gate_centers)
history_sample = np.linspace(0, gate_pca.shape[0] - 1, 8000, dtype=int)
transition_counts = np.asarray(atlas["transition_counts"], dtype=int)
edge_mask = transition_counts >= config.edge_count_threshold
np.fill_diagonal(edge_mask, False)
out_degree = edge_mask.sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), constrained_layout=True)
scatter = axes[0].scatter(gate_pca[history_sample, 0], gate_pca[history_sample, 1],
                          c=atlas["labels"][history_sample], cmap="turbo",
                          vmin=-0.5, vmax=config.clusters - 0.5, s=4, alpha=0.22)
axes[0].scatter(center_pca[:, 0], center_pca[:, 1], c=chart_colors, s=55,
                edgecolor="#111827", linewidth=0.6)
for chart_id, (x, y) in enumerate(center_pca):
    axes[0].annotate(str(chart_id), (x, y), xytext=(2, 2),
                     textcoords="offset points", fontsize=6)
axes[0].set_title("Charts in physical-configuration PCA space")
axes[0].set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")
axes[0].grid(alpha=0.15)
fig.colorbar(scatter, ax=axes[0], label="chart ID", fraction=0.046)

# Show the two strongest observed outgoing transitions per chart. The router
# itself retains every edge meeting the count threshold.
for source in range(config.clusters):
    destinations = np.flatnonzero(edge_mask[source])
    if destinations.size:
        strongest = destinations[np.argsort(transition_counts[source, destinations])[-2:]]
        for destination in strongest:
            strength = transition_counts[source, destination]
            axes[1].annotate("", xy=center_pca[destination], xytext=center_pca[source],
                             arrowprops=dict(arrowstyle="->", color=chart_colors[source],
                                             alpha=min(0.75, 0.18 + strength / 250),
                                             linewidth=0.7))
node_sizes = 35 + 90 * chart_sizes / chart_sizes.max()
axes[1].scatter(center_pca[:, 0], center_pca[:, 1], c=chart_colors, s=node_sizes,
                edgecolor="#111827", linewidth=0.6, zorder=3)
for chart_id, (x, y) in enumerate(center_pca):
    axes[1].annotate(str(chart_id), (x, y), ha="center", va="center", fontsize=5, zorder=4)
axes[1].set_title("Observed transition topology (two strongest per chart)")
axes[1].set_xlabel("PC 1")
axes[1].set_ylabel("PC 2")
axes[1].grid(alpha=0.15)

chart_ids = np.arange(config.clusters)
axes[2].bar(chart_ids, chart_sizes, color=chart_colors, width=0.9, label="offline windows")
axes[2].set_xlabel("chart ID")
axes[2].set_ylabel("offline windows")
axes[2].set_title("Chart support and graph connectivity")
axes[2].grid(axis="y", alpha=0.2)
degree_axis = axes[2].twinx()
degree_axis.plot(chart_ids, out_degree, color="#111827", linewidth=1.2,
                 marker=".", markersize=3, label="out-degree")
degree_axis.set_ylabel("outgoing edges with count >= 3")
lines_1, labels_1 = axes[2].get_legend_handles_labels()
lines_2, labels_2 = degree_axis.get_legend_handles_labels()
axes[2].legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right", fontsize=8)

topology_plot_path = RESULT_ROOT / "physical48_atlas_topology.png"
fig.savefig(topology_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("Active clustering coordinates:", gate_features.shape[1])
print("PCA variance captured:", f"{100 * pca.explained_variance_ratio_.sum():.1f}%")
print("Directed routing edges retained by the controller:", int(edge_mask.sum()))
print("Outgoing-degree range:", int(out_degree.min()), "to", int(out_degree.max()))


### How to read these views

- The shallow arm clouds show posture variation inside each chart, while the bold arm is the medoid. Because assignment uses the current periodic joint configuration, each chart is now directly interpretable as a compact physical-posture neighborhood.
- The workspace plot checks physical coverage and reveals both elbow branches. A medoid is not a target and does not imply that its source trajectory was successful.
- The PCA view is only a two-dimensional projection of the six active configuration coordinates. Overlap in this view does not mean the original chart assignments overlap.
- Node size and support bars show how many complete input-output Hankel columns back each chart. The topology panel draws only two strong outgoing edges per node; online routing uses every observed directed edge with count at least three.


## 3. Fairness audit

The data library contains only random-action episodes. Goal construction excludes every candidate from an offline episode whose environment target is within 6 cm of the online target. The online controller uses only the fixed offline atlas: it does not retrieve trajectories by target identity, fit dynamics, invoke inverse kinematics, or use a feedback fallback.

In [ ]:
primary_runs = dev_summary["results"] + evaluation_runs
audit = evaluation_summary["fairness_audit"]
minimum_source_target_separation = min(
    row["data_goal_source_target_distance"] for row in primary_runs
)

assert audit["offline_stabilized_episode_count"] == 0
assert minimum_source_target_separation >= config.goal_source_target_min_distance
assert all(row["fitted_dynamics_model"] is False for row in primary_runs)
assert all(row["online_nearest_trajectory_retrieval"] is False for row in primary_runs)
assert all(row["use_ik_ref"] is False for row in primary_runs)
assert all(row["feedback_fallback"] is False for row in primary_runs)

pd.Series({
    "offline seed range": str(audit["data_seed_range"]),
    "stabilized offline episodes": audit["offline_stabilized_episode_count"],
    "best offline final-tail mean (m)": audit["offline_best_tail_mean"],
    "minimum goal-source target separation (m)": minimum_source_target_separation,
    "required source-target separation (m)": config.goal_source_target_min_distance,
    "online trajectory retrieval": "none",
    "IK / feedback / fallback": "none",
})

## 4. Baseline evaluation results

Strict success requires both the final-30 and final-100 windows to have mean distance below 1 cm and maximum distance below 2 cm. The table below evaluates the frozen 48-chart Atlas-DeePC controller on every seed currently saved in the baseline summary. Only ranges not previously inspected should be called pristine held-out tests.


In [ ]:
def strict_success(result):
    return bool(result["success_30"] and result["success_100"])

def wilson_interval(successes, trials, z=1.959963984540054):
    rate = successes / trials
    denominator = 1 + z * z / trials
    center = (rate + z * z / (2 * trials)) / denominator
    half_width = z * np.sqrt(rate * (1 - rate) / trials + z * z / (4 * trials**2)) / denominator
    return center - half_width, center + half_width

results = pd.DataFrame([{
    "seed": result["seed"],
    "final-30 mean (mm)": 1000 * result["mean_last_30"],
    "final-100 mean (mm)": 1000 * result["mean_last_100"],
    "minimum (mm)": 1000 * result["minimum"],
    "strict success": strict_success(result),
    "optimal solves": result["statuses"].get("optimal", 0),
} for result in evaluation_runs])
successes = int(results["strict success"].sum())
ci_low, ci_high = wilson_interval(successes, len(results))
summary_table = pd.DataFrame([{
    "method": "Graph-routed Atlas-DeePC",
    "seeds": len(results),
    "successes": successes,
    "success rate": successes / len(results),
    "95% Wilson CI": f"[{ci_low:.3f}, {ci_high:.3f}]",
}])
display(summary_table.style.format({"success rate": "{:.1%}"}))

failures = results.loc[~results["strict success"]].copy()
failures["failure type"] = np.select(
    [failures["minimum (mm)"] < 10, failures["minimum (mm)"] < 20],
    ["reached <1 cm; did not stabilize", "reached <2 cm; did not stabilize"],
    default="never reached 2 cm",
)
display(failures[["seed", "failure type", "minimum (mm)", "final-30 mean (mm)", "final-100 mean (mm)"]].style.format({
    "minimum (mm)": "{:.2f}",
    "final-30 mean (mm)": "{:.2f}",
    "final-100 mean (mm)": "{:.2f}",
}))
print("Expanded estimate:", f"{successes}/{len(results)} = {successes / len(results):.1%}")
print("95% Wilson interval:", f"[{100 * ci_low:.1f}%, {100 * ci_high:.1f}%]")
print("Optimal evaluation solves:", int(results["optimal solves"].sum()), "/", len(results) * config.steps)
print("Median successful final-30 error:", f"{results.loc[results['strict success'], 'final-30 mean (mm)'].median():.2f} mm")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
colors = np.where(results["strict success"], "#15803d", "#b91c1c")
axes[0].scatter(results["seed"], results["final-30 mean (mm)"], c=colors, s=30)
axes[0].axhline(10, color="#111827", linestyle="--", linewidth=1, label="mean threshold (10 mm)")
axes[0].set_yscale("log")
axes[0].set_xlabel("evaluation seed")
axes[0].set_ylabel("final-30 mean distance (mm, log scale)")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

failure_counts = pd.Series({
    "stabilized": successes,
    "reached <1 cm; unstable": int((failures["minimum (mm)"] < 10).sum()),
    "reached <2 cm; unstable": int(((failures["minimum (mm)"] >= 10) & (failures["minimum (mm)"] < 20)).sum()),
    "never reached 2 cm": int((failures["minimum (mm)"] >= 20).sum()),
})
axes[1].bar(failure_counts.index, failure_counts.values, color=["#15803d", "#d97706", "#f59e0b", "#b91c1c"])
axes[1].set_ylabel("number of evaluation seeds")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.2)
for index, value in enumerate(failure_counts.values):
    axes[1].text(index, value + 1, str(value), ha="center")
fig.suptitle(f"Graph-routed Atlas-DeePC: {successes}/{len(results)} successes (95% CI {100*ci_low:.1f}-{100*ci_high:.1f}%)")
plot_path = RESULT_ROOT / f"baseline_atlas_{len(results)}_statistics.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()


## 5. Atlas routing replays

The numerical result uses the full baseline panel reported above. These videos replay the same frozen **48-chart Atlas-DeePC controller** and visualize physical motion, graph routing, chart transitions, target convergence, and applied DeePC inputs.


In [ ]:
def video_gallery(summary, experiment_dir, seeds, title):
    cards = []
    selected = {int(seed) for seed in seeds}
    for result in summary["results"]:
        seed = int(result["seed"])
        if seed not in selected:
            continue
        video_path = (experiment_dir / f"seed{seed}_control.mp4").resolve()
        audit_path = experiment_dir / f"seed{seed}_control_video_verification.json"
        video_audit = json.loads(audit_path.read_text())
        assert video_audit["maximum_replay_output_error"] <= 1e-8
        assert video_path.exists()
        success = result["success_30"] and result["success_100"]
        status = "SUCCESS" if success else "FAILURE"
        color = "#166534" if success else "#b91c1c"
        mean_mm = 1000 * result["mean_last_30"]
        target = summary["fairness_audit"]["held_out_targets"][str(seed)]["target"]
        cards.append(f"""
        <div style="border:1px solid #d1d5db; padding:10px; background:#ffffff;">
          <div style="font:600 14px sans-serif; margin-bottom:6px; color:#111827;">
            Seed {seed} &nbsp; <span style="color:{color}">{status}</span>
          </div>
          <div style="font:12px sans-serif; margin-bottom:8px; color:#4b5563;">
            Final-30 mean: {mean_mm:.3f} mm &nbsp; Target: ({target[0]:+.3f}, {target[1]:+.3f})
          </div>
          <video src="{video_path}" controls loop preload="metadata" style="width:100%; max-width:360px;">
            Your browser does not support the video element.
          </video>
        </div>
        """)
    return HTML(f"""
    <h3 style="font-family:sans-serif">{title}</h3>
    <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:12px;">
      {''.join(cards)}
    </div>
    """)

success_video_seeds = [215, 219, 222, 242, 250, 258, 268, 273, 294, 302]
diagnostic_video_seeds = [211, 232, 277]
print("All 13 displayed videos replay the saved nominal physical-chart rollouts exactly.")
display(video_gallery(video_summary, VIDEO_DIR, success_video_seeds, "Ten representative nominal-Atlas successes"))
display(video_gallery(video_summary, VIDEO_DIR, diagnostic_video_seeds, "Nominal Atlas success and failure diagnostics"))


## 6. Slowed Atlas-routing dashboard

This synchronized visualization uses the saved successful Atlas-DeePC rollout for seed 222. It retains the first 60 control steps and displays them at **0.25x play speed**, so the arm motion and physical-chart changes can be inspected.

The left panel overlays the measured arm, one-step DeePC prediction, active chart-medoid waypoint posture, target, and graph route. The upper-right panel compares the chart requested by the graph with the chart measured at the following control step. A mismatch is labeled `replan`: the route is recomputed from the newly measured physical chart. The lower panels show target convergence and the two applied DeePC inputs.


In [ ]:
dashboard_video = (RESULT_ROOT / "physical48_dashboard_seed222_025x.mp4").resolve()
dashboard_poster = (RESULT_ROOT / "physical48_dashboard_seed222_025x_poster.png").resolve()
dashboard_metadata_path = RESULT_ROOT / "physical48_dashboard_seed222_025x.json"
assert dashboard_video.exists() and dashboard_poster.exists() and dashboard_metadata_path.exists()
dashboard_metadata = json.loads(dashboard_metadata_path.read_text())
display(HTML(f"""
<video src="{dashboard_video}" poster="{dashboard_poster}" controls preload="metadata"
       style="width:100%; max-width:1280px; border:1px solid #d1d5db; background:#f5f7fa;">
  Your browser does not support the video element.
</video>
"""))
print(
    f"Untouched seed {dashboard_metadata['seed']} | "
    f"{dashboard_metadata['play_speed']:.2f}x play speed | "
    f"{dashboard_metadata['displayed_control_steps']} displayed steps | "
    f"final displayed error {dashboard_metadata['final_error_mm']:.3f} mm"
)


In [ ]:
import base64

additional_dashboard_seeds = [211, 213, 237, 269, 302]
video_by_seed = {int(row["seed"]): row for row in video_summary["results"]}

def embedded_media_uri(path, mime_type):
    payload = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{payload}"

cards = []
for seed in additional_dashboard_seeds:
    video_path = RESULT_ROOT / f"physical48_dashboard_seed{seed}_025x.mp4"
    poster_path = RESULT_ROOT / f"physical48_dashboard_seed{seed}_025x_poster.png"
    metadata_path = RESULT_ROOT / f"physical48_dashboard_seed{seed}_025x.json"
    assert video_path.exists() and poster_path.exists() and metadata_path.exists()
    metadata = json.loads(metadata_path.read_text())
    assert metadata["seed"] == seed
    assert "clustering_jointconfig48_fresh211_310" in metadata["source_result"]
    result = video_by_seed[seed]
    mean_mm = 1000 * result["mean_last_30"]
    video_uri = embedded_media_uri(video_path, "video/mp4")
    poster_uri = embedded_media_uri(poster_path, "image/png")
    cards.append(f"""
    <div style="border:1px solid #d1d5db; padding:10px; background:#fff; color:#111827;">
      <div style="font:600 14px sans-serif; margin-bottom:8px;">
        Seed {seed} &nbsp; <span style="font-weight:400; color:#4b5563;">final-30 mean: {mean_mm:.3f} mm</span>
      </div>
      <video src="{video_uri}" poster="{poster_uri}" controls preload="metadata"
             style="display:block; width:100%; background:#f5f7fa;">
        Your browser does not support the video element.
      </video>
    </div>
    """)

display(HTML(f"""
<h3 style="font-family:sans-serif; color:inherit;">Five additional nominal-Atlas dashboards</h3>
<p style="font-family:sans-serif; color:inherit;">
These nominal-controller rollouts use the same frozen 48-chart physical atlas and
<strong>0.25x play speed</strong>. Each dashboard is embedded from its saved nominal
numerical rollout, so it remains playable when the notebook is reopened.
</p>
<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(520px,1fr)); gap:16px;">
  {''.join(cards)}
</div>
"""))
print("Embedded five verified physical-chart dashboards:", additional_dashboard_seeds)


### Terminal stabilization failure

Seed 232 reaches **1.819 mm** from the target but does not remain there. The saved video below shows a baseline failure caused by insufficient terminal stabilization: the controller remains chart-local throughout, with no secondary controller or recovery mode.


In [ ]:
failure_video = (RESULT_ROOT / "physical48_dashboard_seed232_failure_025x.mp4").resolve()
failure_poster = (RESULT_ROOT / "physical48_dashboard_seed232_failure_025x_poster.png").resolve()
failure_metadata_path = RESULT_ROOT / "physical48_dashboard_seed232_failure_025x.json"
failure_result_path = ATLAS_DIR / "seed232_control_results.npz"
assert all(path.exists() for path in [failure_video, failure_poster, failure_metadata_path, failure_result_path])
failure_metadata = json.loads(failure_metadata_path.read_text())
failure_result = np.load(failure_result_path, allow_pickle=True)
failure_distance = np.asarray(failure_result["distance"], dtype=float)
failure_summary = json.loads(failure_result["summary"].item())
failure_closest_step = int(np.argmin(failure_distance))
display(HTML(f"""
<video src="{failure_video}" poster="{failure_poster}" controls preload="metadata"
       style="width:100%; max-width:1280px; border:1px solid #b91c1c; background:#f5f7fa;">
  Your browser does not support the video element.
</video>
"""))
print(
    f"Untouched seed {failure_metadata['seed']} | "
    f"{failure_metadata['play_speed']:.2f}x play speed | "
    f"closest error {1000 * failure_distance[failure_closest_step]:.3f} mm at step {failure_closest_step + 1} | "
    f"final displayed error {failure_metadata['final_error_mm']:.3f} mm | "
    f"baseline mean-100 {1000 * failure_summary['mean_last_100']:.3f} mm"
)


### Mechanical joint-limit failure

Untouched seed 277 reproduces the same mechanism as the former seed-116 failure, mirrored at the opposite joint limit. The graph routes through charts `30 -> 39 -> 11`. While chart 39's waypoint has $q_2=+167.5^\circ$, the exact target posture associated with goal chart 11 has $q_2=-154.8^\circ$. When the 7 cm latch activates at step 29, the reference therefore jumps from the chart-39 waypoint to the target posture.

In ordinary joint coordinates this change requires a large negative rotation, but the periodic sine/cosine representation reports the wrapped displacement as a small positive rotation. DeePC follows that apparent short direction: the measured second joint reaches **$+172.1^\circ$**, approximately the upper mechanical stop, while the second input saturates at **$+1.0$** and the joint velocity falls to zero. The fingertip's closest approach is **40.604 mm at step 34**, and its final-30 mean is **58.574 mm**.

The periodic chart geometry treats the two sides of $\pm\pi$ as adjacent, whereas the mechanically limited joint cannot pass through that boundary. This exposes a limitation of the baseline graph: observed chart adjacency alone does not guarantee that the selected waypoint transition is mechanically feasible from the current posture.


In [ ]:
unreached_video = (RESULT_ROOT / "physical48_dashboard_seed277_unreached_025x.mp4").resolve()
unreached_poster = (RESULT_ROOT / "physical48_dashboard_seed277_unreached_025x_poster.png").resolve()
unreached_metadata_path = RESULT_ROOT / "physical48_dashboard_seed277_unreached_025x.json"
unreached_result_path = ATLAS_DIR / "seed277_control_results.npz"
assert all(path.exists() for path in [unreached_video, unreached_poster, unreached_metadata_path, unreached_result_path])
unreached_metadata = json.loads(unreached_metadata_path.read_text())
unreached_result = np.load(unreached_result_path, allow_pickle=True)
unreached_distance = np.asarray(unreached_result["distance"], dtype=float)
unreached_summary = json.loads(unreached_result["summary"].item())
unreached_closest_step = int(np.argmin(unreached_distance))
display(HTML(f"""
<video src="{unreached_video}" poster="{unreached_poster}" controls preload="metadata"
       style="width:100%; max-width:1280px; border:1px solid #b91c1c; background:#f5f7fa;">
  Your browser does not support the video element.
</video>
"""))
print(
    f"Untouched seed {unreached_metadata['seed']} | "
    f"{unreached_metadata['play_speed']:.2f}x play speed | "
    f"closest error {1000 * unreached_distance[unreached_closest_step]:.3f} mm at step {unreached_closest_step + 1} | "
    f"final displayed error {unreached_metadata['final_error_mm']:.3f} mm | "
    f"baseline mean-100 {1000 * unreached_summary['mean_last_100']:.3f} mm"
)


## 7. Reproduce the baseline controller evaluation

The cell below is disabled by default because evaluation rollouts are already saved. Change the single seed-range line as needed. Setting `RERUN=True` evaluates graph-routed Atlas-DeePC with the frozen 48-chart atlas, rank-60 chart matrices, and 6 cm fairness guard. No videos are rendered.


In [ ]:
RERUN = False
EVALUATION_SEEDS = list(range(211, 711))  # Change only this line; Python stop is exclusive.
REPRODUCTION_DIR = RESULT_ROOT / "baseline_atlas_reproduction"
OVERWRITE = False
if RERUN:
    REPRODUCTION_DIR.mkdir(parents=True, exist_ok=True)
    experiment.save_atlas(REPRODUCTION_DIR / "behavioral_atlas.npz", atlas, config)
    rerun_results = []
    for seed in EVALUATION_SEEDS:
        result_path = REPRODUCTION_DIR / f"seed{seed}_control_results.npz"
        if result_path.exists() and not OVERWRITE:
            summary = json.loads(np.load(result_path, allow_pickle=True)["summary"].item())
        else:
            run_config = experiment.RoutedDeePCConfig(**{**asdict(config), "seed": seed})
            summary = experiment.run_routed_deepc(H, atlas, run_config, save_path=result_path)
        summary["result_path"] = str(result_path)
        rerun_results.append(summary)
        print(f"seed {seed}: strict success = {strict_success(summary)}", flush=True)
    rerun_audit = experiment.audit_iid_library(H, eval_seeds=EVALUATION_SEEDS)
    rerun_payload = {
        "method": "graph_routed_fixed_chart_deepc",
        "config": asdict(config),
        "fairness_audit": rerun_audit,
        "chart_sizes": np.bincount(atlas["labels"].astype(int), minlength=config.clusters).tolist(),
        "results": rerun_results,
    }
    (REPRODUCTION_DIR / "summary.json").write_text(json.dumps(rerun_payload, indent=2))
else:
    print("Using the saved baseline evaluation. Set RERUN=True to run the requested seeds.")


## Interpretation

Graph-routed Atlas-DeePC uses a physically interpretable gate: 48 charts partition the arm's latest periodic joint configuration. The current chart supplies a cached rank-60 complete input-output behavioral matrix, while the offline transition graph supplies only waypoint references.

The controller never changes to an online-selected matrix. At each step it localizes the measured posture, routes toward the goal chart, and solves the same DeePC QP using the active chart's fixed behavioral basis. The table and plots above compute the baseline success rate and failure composition from the complete evaluation panel currently saved on disk.

The method remains purely data-enabled predictive control: it does not fit dynamics or invoke inverse kinematics, contextual rescue, or a feedback fallback. Remaining failures include terminal drift and mechanically infeasible graph routes. Future changes should be developed outside the reported seeds and tested on a newly frozen range.
